# 🎙️ Fish Audio Drama 3 (35B-A3B) + Fish Speech WebUI в Google Colab (A100 40/80GB & G4 96GB)

Оптимизированный ноутбук «всё в одном» для запуска флагманской модели синтеза речи, клонирования голоса и многоголосой озвучки сцен **[Fish Audio Drama 3 (35B-A3B)](https://huggingface.co/fishaudio/drama-3)** с официальным веб-интерфейсом **[Fish Speech Awesome WebUI](https://github.com/fishaudio/fish-speech)**, адаптированный сразу под **A100 (40 ГБ / 80 ГБ)** и **G4 (96 ГБ VRAM)** в Google Colab.

### 💡 Умная автооптимизация под A100 и G4 (`PRECISION_MODE = "AUTO"`)
- **🎯 A100 40GB (SM80)**:
  - Автоматически включает профиль **`W8A16_SAFE` без CUDA Graphs** (`expandable_segments:True`), удерживая все 256 экспертов MoE в компактном FP8 в видеопамяти и распаковывая активные блоки налету в BF16 — модель гарантированно помещается в **39.56 ГБ VRAM** без `CUDA Out of Memory`!
- **⚡ A100 80GB (SM80) и G4 96GB (SM89+)**:
  - Автоматически включает полный стек **CUDA Graphs (`1024 → 10240` токенов)** (~42 ГБ VRAM) и аппаратные FP8 Tensor Cores (на SM89+) либо быстрый `W8A16 + Graphs` (на A100 80GB). Также доступен режим **`BF16_ULTRA`** (~76 ГБ VRAM).
- **🖥️ Официальный интерфейс Fish Speech Awesome WebUI**:
  - Автоматическая сборка современного TypeScript-интерфейса (`awesome_webui`) на Node.js 22.
- **💾 Полная интеграция с Google Drive**:
  - Веса модели (`~41 ГБ`) кэшируются в `MyDrive/FishSpeech_Models/drama-3` (скачиваются только при первом запуске).
  - Ваши референсные голоса сохраняются в `MyDrive/FishSpeech_References`, а сгенерированные записи — в `MyDrive/FishSpeech_Outputs`.
- **🚀 3 быстрых способа подключения к WebUI и API**:
  - **Google Colab Proxy**, **Pinggy Tunnel** и **Cloudflare Tunnel (HTTP/2)** выдаются одновременно.

> ⚠️ **Внимание**: Перед запуском убедитесь, что в верхнем меню **«Среда выполнения» (Runtime) → «Сменить среду выполнения» (Change runtime type)** выбран ускоритель **A100 GPU** или **G4 GPU**.

## Step 1: Проверка окружения (A100 / G4 GPU) и подключение Google Drive

In [ ]:
# Проверяем выделенный GPU, архитектуру CUDA и объём видеопамяти (ожидается ~96 ГБ)
!nvidia-smi
import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"🔥 GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.1f} ГБ | Compute Capability: SM{props.major}{props.minor}")

# Сохранение кэша модели, референсных голосов и результатов на Google Drive
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive успешно подключён!")

## Step 2: Установка Fish Speech, сборка Awesome WebUI и оптимизация под G4 96GB

- Устанавливает аудио-библиотеки (`portaudio`, `sox`, `ffmpeg`) и зависимости `fishaudio/fish-speech`.
- Собирает официальный фронтенд **Awesome WebUI**.
- Настраивает поддержку аппаратного **FP8 (W8A8)** и безопасного фоллбэка **W8A16** для всех поколений GPU.

In [ ]:
import os
from pathlib import Path

print("📦 Установка системных аудио-библиотек (portaudio, sox, ffmpeg) и Node.js 22 LTS...")
!apt-get update -qq && apt-get install -y -qq portaudio19-dev libsox-dev ffmpeg xz-utils

# Устанавливаем свежий Node.js v22.14.0 (требуется Node 22.12+ для Vite и @tailwindcss/oxide)
!wget -q https://nodejs.org/dist/v22.14.0/node-v22.14.0-linux-x64.tar.xz -O /tmp/node22.tar.xz
!tar -xJf /tmp/node22.tar.xz -C /usr/local --strip-components=1
!rm -f /tmp/node22.tar.xz
!node -v && npm -v

%cd /content
if not os.path.exists("/content/fish-speech"):
    !git clone https://github.com/fishaudio/fish-speech.git
else:
    %cd /content/fish-speech
    !git pull

%cd /content/fish-speech
# Убираем из pyproject.toml жёсткий override protobuf<6.0.0, который ломает TensorBoard в Colab
pyproject_path = Path("/content/fish-speech/pyproject.toml")
if pyproject_path.exists():
    pyproject_path.write_text(
        pyproject_path.read_text(encoding="utf-8").replace("protobuf>=3.20.0,<6.0.0", "protobuf>=6.31.1"),
        encoding="utf-8"
    )

print("⚡ Быстрая установка Python-зависимостей через uv (без конфликта версий protobuf/torch)...")
!pip install -q uv
!uv pip install --system --no-config -q "huggingface_hub" "transformers>=5.0.0" "tokenizers>=0.21.0" \
    "hydra-core>=1.3.2" "natsort>=8.4.0" "einops>=0.7.0" "librosa>=0.10.1" \
    "kui>=1.6.0" "uvicorn>=0.30.0" "loguru>=0.6.0" "pyrootutils>=1.0.4" \
    "resampy>=0.4.3" "einx==0.2.2" pydub pyaudio "modelscope>=1.17.1" \
    "opencc-python-reimplemented==0.1.7" silero-vad ormsgpack "tiktoken>=0.8.0" \
    "pydantic>=2.9.2" cachetools safetensors "soundfile>=0.12.1" \
    "protobuf>=6.31.1" loralib julius argbind randomname markdown2 \
    flatten-dict pyloudnorm ffmpy pystoi torch-stoi importlib-resources
!uv pip install --system --no-config -q --no-deps descript-audio-codec descript-audiotools -e .
!uv pip install --system --no-config -q --no-deps --upgrade "protobuf>=6.31.1"

# Отключаем ненужный импорт TensorBoard внутри descript-audiotools
import site
for sp in site.getsitepackages():
    dec_file = Path(sp) / "audiotools" / "ml" / "decorators.py"
    if dec_file.exists():
        dec_file.write_text(
            dec_file.read_text(encoding="utf-8").replace(
                "from torch.utils.tensorboard import SummaryWriter",
                "SummaryWriter = None"
            ),
            encoding="utf-8"
        )

# Сборка официального веб-интерфейса Awesome WebUI на Node.js 22
webui_dist = Path("/content/fish-speech/awesome_webui/dist/index.html")
if not webui_dist.exists():
    print("🛠️ Сборка интерфейса Fish Speech Awesome WebUI (Node 22)...")
    %cd /content/fish-speech/awesome_webui
    !rm -rf node_modules package-lock.json
    !npm install --no-audit --no-fund
    !npm run build
    %cd /content/fish-speech
print(f"✅ WebUI собран: {webui_dist.exists()}")

# Патч: Универсальная поддержка FP8 на A100 (SM80 W8A16) и G4 (SM89/SM120 W8A8 Tensor Cores)
fp8_path = Path("/content/fish-speech/fish_speech/models/text2semantic/fish_omni/fp8_moe.py")
moe_path = Path("/content/fish-speech/fish_speech/models/text2semantic/fish_omni/moe.py")
if fp8_path.exists() and moe_path.exists():
    fp8_code = fp8_path.read_text(encoding="utf-8")
    fp8_code = fp8_code.replace(
        "torch.cuda.get_device_capability(device) >= (8, 9)",
        "torch.cuda.get_device_capability(device) >= (8, 0)"
    )
    fp8_path.write_text(fp8_code, encoding="utf-8")

    moe_code = moe_path.read_text(encoding="utf-8")
    if "FISH_DRAMA3_W8A16_FALLBACK" not in moe_code:
        moe_code = moe_code.replace(
            "if self.fp8 and x.is_cuda:",
            "if self.fp8 and x.is_cuda and torch.cuda.get_device_capability(x.device) >= (8, 9) and os.environ.get('FISH_DRAMA3_W8A16_FALLBACK', '0') != '1':"
        )
        if "import os" not in moe_code:
            moe_code = "import os\n" + moe_code
        moe_path.write_text(moe_code, encoding="utf-8")
    print("🚀 Настроена поддержка W8A16 (для A100 SM80) и FP8 Tensor Cores (для G4 SM89+)!")

print("✅ Установка и настройка Fish Speech завершены!")

## Step 3: Загрузка весов `fishaudio/drama-3` и привязка папок Google Drive

- Если `USE_GOOGLE_DRIVE = True`:
  - **Кэш весов Drama 3**: `/content/drive/MyDrive/FishSpeech_Models/drama-3` (скачивается только при первом запуске)
  - **Референсные голоса**: `/content/drive/MyDrive/FishSpeech_References` (все загруженные в WebUI голоса сохраняются навсегда)
  - **Папка для готовых аудио**: `/content/drive/MyDrive/FishSpeech_Outputs`

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

# Опционально: укажите HF_TOKEN для максимальной скорости скачивания с Hugging Face
HF_TOKEN = ""  # @param {type:"string"}
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()

os.makedirs("/content/fish-speech/checkpoints", exist_ok=True)

if USE_GOOGLE_DRIVE:
    DRIVE_MODELS_DIR = "/content/drive/MyDrive/FishSpeech_Models/drama-3"
    DRIVE_REFS_DIR = "/content/drive/MyDrive/FishSpeech_References"
    DRIVE_OUT_DIR = "/content/drive/MyDrive/FishSpeech_Outputs"
    os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
    os.makedirs(DRIVE_REFS_DIR, exist_ok=True)
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

    # Симлинк для чекпоинта drama-3
    local_ckpt = "/content/fish-speech/checkpoints/drama-3"
    if os.path.exists(local_ckpt) and not os.path.islink(local_ckpt):
        !rm -rf {local_ckpt}
    if not os.path.exists(local_ckpt):
        os.symlink(DRIVE_MODELS_DIR, local_ckpt)

    # Симлинк для референсных голосов
    local_refs = "/content/fish-speech/references"
    if os.path.exists(local_refs) and not os.path.islink(local_refs):
        !rm -rf {local_refs}
    if not os.path.exists(local_refs):
        os.symlink(DRIVE_REFS_DIR, local_refs)

    TARGET_CKPT_DIR = DRIVE_MODELS_DIR
    print(f"📁 Папки синхронизированы с Google Drive:\n  - Модель: {DRIVE_MODELS_DIR}\n  - Референсы: {DRIVE_REFS_DIR}\n  - Вывод: {DRIVE_OUT_DIR}")
else:
    TARGET_CKPT_DIR = "/content/fish-speech/checkpoints/drama-3"
    os.makedirs(TARGET_CKPT_DIR, exist_ok=True)
    os.makedirs("/content/fish-speech/references", exist_ok=True)

# Проверяем, все ли 15 шардов модели и codec.pth уже есть в кэше
ckpt_path = Path(TARGET_CKPT_DIR)
shards = list(ckpt_path.glob("model-*-of-00015.safetensors"))
has_codec = (ckpt_path / "codec.pth").exists()

if len(shards) == 15 and has_codec:
    print("⚡ Все 15 шардов модели fishaudio/drama-3 и codec.pth уже найдены на Google Drive! Скачивание пропускается.")
else:
    print("⬇️ Скачивание весов fishaudio/drama-3 с Hugging Face в Google Drive (~41 ГБ, только при первом запуске)...")
    snapshot_download(
        repo_id="fishaudio/drama-3",
        local_dir=TARGET_CKPT_DIR,
        cache_dir="/content/hf_temp_cache",
        local_dir_use_symlinks=False
    )
    # Удаляем временную папку .cache внутри Google Drive, чтобы не занимать лишнее место на Диске
    !rm -rf "{TARGET_CKPT_DIR}/.cache" /content/hf_temp_cache
    print("✅ Загрузка fishaudio/drama-3 на Google Drive успешно завершена!")

## Step 4: Запуск сервера Fish Speech (Drama 3 на A100 40/80GB и G4 96GB) + Awesome WebUI и 3 туннеля

Автоматическая оптимизация (`PRECISION_MODE = "AUTO"`) сама подбирает идеальный профиль под вашу видеокарту:
- **A100 40GB**: `W8A16_SAFE` без CUDA Graphs, чтобы гарантированно уложиться в 39.56 ГБ VRAM без OOM.
- **A100 80GB / G4 96GB**: `FP8 / W8A16` + полный стек CUDA Graphs (~42 ГБ VRAM, максимальная скорость генерации). Также доступен режим `"BF16_ULTRA"` (~76 ГБ VRAM).

Сразу после прогрева модели появятся **3 быстрые ссылки для входа в WebUI и API**:
1. 🖥️ **Google Colab Proxy** — быстрый вход в браузере.
2. ⚡ **Pinggy Tunnel** — быстрый SSH-туннель без Cloudflare.
3. 🌐 **Cloudflare Tunnel (HTTP/2)** — основная ссылка для API (`client.py`) и резервный WebUI.

In [ ]:
import os
import subprocess
import threading
import time
import re
import site
import socket
from pathlib import Path
import torch

# Режим для A100: "AUTO" сам подберёт идеальные настройки под 40 ГБ или 80 ГБ
# Можно также принудительно задать "W8A16_SAFE" или "BF16_ULTRA" (только для A100 80GB)
PRECISION_MODE = "AUTO"

# Останавливаем старые процессы при перезапуске ячейки
!pkill -9 -f "tools/api_server.py" 2>/dev/null || true
!pkill -9 -f "cloudflared" 2>/dev/null || true
time.sleep(1)

# Ищем свободный порт (8080 занят самим Colab)
PORT = 7860
for candidate_port in (7860, 7861, 8189, 8288):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", candidate_port)) != 0:
            PORT = candidate_port
            break
print(f"🔌 Выбран свободный порт для сервера: {PORT}")

# Проверяем сборку Awesome WebUI
webui_dist = Path("/content/fish-speech/awesome_webui/dist/index.html")
if not webui_dist.exists():
    print("🛠️ Собираем фронтенд Awesome WebUI (~30 сек)...")
    !wget -q https://nodejs.org/dist/v22.14.0/node-v22.14.0-linux-x64.tar.xz -O /tmp/node22.tar.xz
    !tar -xJf /tmp/node22.tar.xz -C /usr/local --strip-components=1
    !rm -f /tmp/node22.tar.xz
    %cd /content/fish-speech/awesome_webui
    !rm -rf node_modules package-lock.json
    !npm install --no-audit --no-fund
    !npm run build
    %cd /content/fish-speech

# Фикс маршрута WebUI и зависимостей protobuf/tensorboard
views_path = Path("/content/fish-speech/tools/server/views.py")
if views_path.exists():
    v_code = views_path.read_text(encoding="utf-8")
    v_code = v_code.replace('@routes.http("/")\n@routes.http("/ui")\nclass WebUI(HttpView):', '@routes.http("/ui")\nclass WebUI(HttpView):')
    views_path.write_text(v_code, encoding="utf-8")

pyproject_path = Path("/content/fish-speech/pyproject.toml")
if pyproject_path.exists():
    pyproject_path.write_text(
        pyproject_path.read_text(encoding="utf-8").replace("protobuf>=3.20.0,<6.0.0", "protobuf>=6.31.1"),
        encoding="utf-8"
    )
!uv pip install --system --no-config -q flatten-dict pyloudnorm ffmpy pystoi torch-stoi importlib-resources
!uv pip install --system --no-config -q --no-deps --upgrade "protobuf>=6.31.1"
for sp in site.getsitepackages():
    dec_file = Path(sp) / "audiotools" / "ml" / "decorators.py"
    if dec_file.exists():
        dec_file.write_text(
            dec_file.read_text(encoding="utf-8").replace(
                "from torch.utils.tensorboard import SummaryWriter",
                "SummaryWriter = None"
            ),
            encoding="utf-8"
        )

# Скачиваем Cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# --- АВТОКОНФИГ ПОД A100 (40GB или 80GB) ---
assert torch.cuda.is_available(), "❌ GPU не найден! Включи A100 в Runtime -> Change runtime type"
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024**3)
is_sm89_plus = (props.major, props.minor) >= (8, 9)

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["PYTHONWARNINGS"] = "ignore::SyntaxWarning"

if PRECISION_MODE == "AUTO":
    if vram_gb < 50:
        # Профиль A100 40GB: W8A16 без CUDA Graphs, чтобы уложиться в 39.56 ГБ VRAM без OOM
        active_mode = "W8A16_SAFE (A100 40GB)"
        enable_graphs = False
        env["FISH_DRAMA3_FP8_EXPERTS"] = "1"
        env["FISH_DRAMA3_W8A16_FALLBACK"] = "1"
    else:
        # Профиль A100 80GB: W8A16 + полные CUDA Graphs (~42 ГБ из 80 ГБ, максимальная скорость)
        active_mode = "W8A16_SAFE + Graphs (A100 80GB)"
        enable_graphs = True
        env["FISH_DRAMA3_FP8_EXPERTS"] = "1"
        env["FISH_DRAMA3_W8A16_FALLBACK"] = "0" if is_sm89_plus else "1"
elif PRECISION_MODE == "BF16_ULTRA":
    active_mode = "BF16_ULTRA"
    enable_graphs = vram_gb > 85
    env["FISH_DRAMA3_FP8_EXPERTS"] = "0"
    env["FISH_DRAMA3_W8A16_FALLBACK"] = "0"
else:
    active_mode = "W8A16_SAFE"
    enable_graphs = vram_gb >= 50
    env["FISH_DRAMA3_FP8_EXPERTS"] = "1"
    env["FISH_DRAMA3_W8A16_FALLBACK"] = "0" if is_sm89_plus else "1"

env["FISH_DRAMA3_CUDA_GRAPH"] = "1" if enable_graphs else "0"

print(f"⚡ Запуск Fish Speech (Drama 3) на {props.name} ({vram_gb:.1f} ГБ VRAM, SM{props.major}{props.minor})")
print(f"   Активный профиль: {active_mode} | CUDA Graphs: {'Вкл' if enable_graphs else 'Выкл'}")

%cd /content/fish-speech
api_proc = subprocess.Popen([
    "python", "tools/api_server.py",
    "--listen", f"127.0.0.1:{PORT}",
    "--llama-checkpoint-path", "checkpoints/drama-3",
    "--device", "cuda:0"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)

server_ready = threading.Event()

def log_server():
    for line in iter(api_proc.stdout.readline, ''):
        print("[FishSpeech]", line, end="")
        if "Uvicorn running on" in line or "Application startup complete" in line:
            server_ready.set()

threading.Thread(target=log_server, daemon=True).start()

print("⏳ Загрузка весов Drama 3 в VRAM (обычно 2–4 мин)...")
server_ready.wait(timeout=1200)
time.sleep(2)

# 1. Встроенный прокси Google Colab (для браузера)
try:
    from google.colab.output import eval_js
    colab_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
except Exception:
    colab_url = None

# 2. Pinggy SSH-туннель
pinggy_proc = subprocess.Popen([
    "ssh", "-p", "443",
    f"-R0:127.0.0.1:{PORT}",
    "-o", "StrictHostKeyChecking=no",
    "-o", "ServerAliveInterval=30",
    "a.pinggy.io"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

pinggy_url = None
def watch_pinggy():
    global pinggy_url
    for line in iter(pinggy_proc.stdout.readline, ''):
        m = re.search(r'https://[a-zA-Z0-9-]+\.a\.free\.pinggy\.link', line)
        if m and not pinggy_url:
            pinggy_url = m.group(0)
            print(f"\n⚡ [Pinggy] WebUI: {pinggy_url}/ui  |  API: {pinggy_url}")

threading.Thread(target=watch_pinggy, daemon=True).start()

# 3. Cloudflare Tunnel (основной для API)
tunnel_proc = subprocess.Popen([
    "/content/cloudflared", "tunnel",
    "--url", f"http://127.0.0.1:{PORT}",
    "--protocol", "http2"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

cf_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match and not cf_url:
        cf_url = match.group(0)
        time.sleep(2)
        print("\n" + "="*72)
        print(f"🎉 DRAMA 3 ГОТОВ НА {props.name} ({vram_gb:.1f} ГБ VRAM)!")
        if colab_url:
            print(f"🖥️  WebUI (Colab Proxy): {colab_url.rstrip('/')}/ui")
        if pinggy_url:
            print(f"⚡ WebUI (Pinggy):      {pinggy_url}/ui")
        print(f"🌐 WebUI (Cloudflare):  {cf_url}/ui")
        print("-" * 72)
        print(f"📡 ССЫЛКА ДЛЯ API (вставлять в client.py): {cf_url}")
        print("="*72 + "\n")
        break

def drain_cf():
    for _ in iter(tunnel_proc.stdout.readline, ''):
        pass

threading.Thread(target=drain_cf, daemon=True).start()

try:
    api_proc.wait()
except KeyboardInterrupt:
    print("\n⏹️ Сервер остановлен.")
    api_proc.terminate()
    tunnel_proc.terminate()
    pinggy_proc.terminate()

## 🛑 (После завершения работы) Как гарантированно остановить списание CU

Когда закончите озвучку и тесты, **чтобы со 100% надёжностью остановить расход Compute Units (CU)**:
1. Нажмите **«Среда выполнения» (Runtime)** в верхнем меню Colab.
2. Выберите **«Отключиться и удалить среду выполнения» (Disconnect and delete runtime)**.